# 01 - LLaDA 마스킹과 가중 손실

**학습 목표**: `t ~ U(0,1)`에서 토큰을 독립적으로 마스킹하고, 마스크 위치에만 `1/t` 가중 손실을 주는 이유를 작은 예제로 확인합니다. 이 코드는 8B LLaDA를 재현하지 않는 toy입니다.

**실행 방법**: Python 3/Jupyter에서 셀을 위에서 아래 순서로 실행합니다. 외부 패키지는 필요하지 않으며 Python 표준 라이브러리만 사용합니다.

In [ ]:
import math
import random

MASK = '[MASK]'
tokens = 'diffusion can restore several masked tokens'.split()

def corrupt(sequence, t, rng):
    # 각 위치를 같은 확률 t로 독립 마스킹합니다.
    return [MASK if rng.random() < t else token for token in sequence]

rng = random.Random(7)
t = 0.55
noisy = corrupt(tokens, t, rng)
print('clean:', tokens)
print('noisy:', noisy)
assert any(x == MASK for x in noisy)

In [ ]:
def weighted_mask_loss(clean, noisy, correct_probability, t):
    # 실제 모델이라면 각 위치 확률은 Transformer logits에서 옵니다.
    per_token = []
    for original, observed in zip(clean, noisy):
        if observed == MASK:
            per_token.append(-math.log(correct_probability) / t)
    return sum(per_token), per_token

total, terms = weighted_mask_loss(tokens, noisy, 0.72, t)
print('masked positions:', len(terms))
print('1/t weighted loss:', round(total, 4))
assert total > 0

## 해석

`t`가 작으면 마스크 수는 적지만 각 항의 `1/t` 가중치는 커집니다. 기대값을 맞추는 것과 mini-batch 분산이 작다는 것은 별개입니다. `03_advanced`에서 이 분산을 측정합니다.